# Phase 3 · Synthesizer Benchmark (CTAB-GAN+ Protocol)

**Goal**: Compare multiple generative models to see which one learns the real data distribution best. 
We benchmark across 5 architectures following the methodology in the CTAB-GAN+ paper:

1. SMOTE (Synthetic Minority Over-sampling Technique)
2. CTGAN
3. TVAE
4. CopulaGAN
5. TabDDPM (Diffusion)

**Evaluation Protocol**:
- 80/20 train/test split on real data.
- Fit synthesizers on the training set.
- Generate a synthetic set equal in size to the training set.
- Evaluate Statistical Similarity (Average JSD, Average Wasserstein Distance, Correlation Difference).
- Evaluate ML Utility (Accuracy, F1, AUC across 5 classification models evaluated on the real test set).
- Report aggregated metrics across N random seeds.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Cell 1 · Imports & Environment Setup
# ──────────────────────────────────────────────────────────────────────────────
import os, sys, logging, warnings
from pathlib import Path
import pandas as pd

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('phase3_benchmark')

# ── Resolve project root ──────────────────────────────────────────────────────
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    PROJECT_ROOT = Path('/kaggle/working/neural-sentinel')
else:
    PROJECT_ROOT = Path(os.getcwd()).resolve()
    while not (PROJECT_ROOT / 'AGENTS.md').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
        PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim'
OUTPUT_DIR  = PROJECT_ROOT / 'docs' / 'generator_benchmark'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('\n✓ Imports complete')

Project root: /home/red/Desktop/neural-sentinel

✓ Imports complete


## 1. Load Real Dataset for Benchmarking

In [5]:
# We benchmark on the ML-ready feature set derived from real data.
# For this notebook, we'll use a subset if the data is large to speed up evaluation.
# Note: This expects a cleaned real dataset with features and a target.
# Since we are building the pipeline, we'll load the interim canonical transactions
# and construct a small feature set for benchmarking.

from src.generation.core.feature_engineer import engineer_features
from src.generation.core.enricher import enrich_transactions

# Load canoncial data
tx_real = pd.read_parquet(INTERIM_DIR / 'transactions.parquet')
acc_real = pd.read_parquet(INTERIM_DIR / 'accounts.parquet')

# Use a random sample for the benchmark to keep runtime manageable
SAMPLE_SIZE = 10_000
if len(tx_real) > SAMPLE_SIZE:
    tx_sample = tx_real.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
else:
    tx_sample = tx_real.copy()

print(f"Using {len(tx_sample):,} rows for benchmark.")

# Enrich and Feature Engineer (determinstic mapping to ML format)
enriched = enrich_transactions(tx_sample, acc_real)
ml_features_real = engineer_features(enriched)

# Drop columns that are IDs, pure dates, or internal metadata, keeping only ML features
cols_to_drop = ['Sender_account', 'Receiver_account', 'Date', 'Time', 
                'transaction_id', 'sender_account_id', 'receiver_account_id',
                '_validation_passed', '_violation_codes', 'original_currency']

ml_data = ml_features_real.drop(columns=[c for c in cols_to_drop if c in ml_features_real.columns])

TARGET_COL = 'is_suspicious_tx'
print(f"\nBenchmark dataset shape: {ml_data.shape}")
print(f"Columns: {list(ml_data.columns)}")
print(f"Target '{TARGET_COL}' distribution:\n{ml_data[TARGET_COL].value_counts(normalize=True)}")

2026-08-05 21:33:23,061 [INFO] Phase 7: Enriching 10000 transactions...
2026-08-05 21:33:23,106 [INFO] Enrichment complete: 10000 rows, 50 cols
2026-08-05 21:33:23,108 [INFO] Phase 8: Feature engineering on 10000 rows...
2026-08-05 21:33:23,116 [INFO]   Computing velocity features (may take a moment)...


Using 10,000 rows for benchmark.


2026-08-05 21:33:24,030 [INFO] Feature engineering complete. Columns: 63



Benchmark dataset shape: (10000, 58)
Columns: ['transaction_date', 'transaction_time', 'transaction_type', 'amount_npr', 'exchange_rate', 'channel', 'sender_country', 'receiver_country', 'is_cross_border', 'remittance_corridor', 'merchant_category', 'device_type', 'ip_address', 'ip_country', 'ip_is_vpn', 'is_fraud', 'fraud_type', 'aml_risk_indicator', 'hour_of_day', 'day_of_week', 'is_weekend', 'month', 'above_1M_NPR', 'above_10M_NPR', 'velocity_sum_10tx', 'tx_count_10', 'tx_count_30', 'currency_mismatch', 'sender_country_risk', 'receiver_country_risk', 'sender_pep', 'sender_sanctions', 'receiver_pep', 'receiver_sanctions', 'sender_risk_grade', 'sender_account_age_days', 'receiver_account_age_days', 'sender_city', 'sender_account_type', 'sender_kyc_verified', 'receiver_city', 'receiver_account_type', 'receiver_kyc_verified', 'Sender_bank_location', 'Receiver_bank_location', 'fx_rate_to_npr', 'amount_local_npr', 'log_amount', 'amount_zscore', 'transmode_code', 'transmode_A', 'transmode

## 2. Initialize Synthesizers

In [ ]:
from src.generation.synthesizers.ctgan_generator import CTGANGenerator
from src.generation.synthesizers.tvae_generator import TVAEGenerator
from src.generation.synthesizers.copulagan_generator import CopulaGANGenerator
from src.generation.synthesizers.smote_generator import SMOTEGenerator
from src.generation.synthesizers.tabddpm_generator import TabDDPMGenerator
from src.generation.synthesizers.tabsyn_generator import TabSynGenerator
from src.generation.synthesizers.smote_generator import SMOTE
# We instantiate each generator. They all implement the BaseGenerator interface.
# Note: TabDDPM requires synthcity to be installed. We gracefully handle if it's missing.

synthesizers = {
    "SMOTE": SMOTEGenerator(config={'k_neighbors': 5, 'random_state': 42}),
    "CTGAN": CTGANGenerator(config={'epochs': 50}),  # Reduced epochs for speed in benchmark
    "TVAE": TVAEGenerator(config={'epochs': 50}),
    "CopulaGAN": CopulaGANGenerator(config={'epochs': 50}),
}

try:
    synthesizers["TabDDPM"] = TabDDPMGenerator(config={})
except Exception as e:
    print(f"TabDDPM skipped: {e}")
    
print("Synthesizers ready for evaluation:")
for name in synthesizers.keys():
    print(f" - {name}")

Synthesizers ready for evaluation:
 - SMOTE
 - CTGAN
 - TVAE
 - CopulaGAN
 - TabDDPM


## 3. Run Benchmark

In [5]:
from src.evaluation.benchmark_runner import BenchmarkRunner

# Configure the benchmark
# In a real run, you'd use multiple seeds (e.g., [42, 0, 1]). 
# We use [42] here for a faster notebook execution.
runner = BenchmarkRunner(
    real_data=ml_data,
    target_col=TARGET_COL,
    seeds=[42,4,6],
    test_size=0.2,
    output_dir=OUTPUT_DIR
)

# Run the full pipeline
results = runner.run(synthesizers)

print("\n✓ Benchmark complete")

2026-08-05 20:06:06,382 [INFO] BenchmarkRunner: 10000 rows, target=is_suspicious_tx, 9 continuous, 48 categorical
2026-08-05 20:06:06,383 [INFO] ============================================================
2026-08-05 20:06:06,383 [INFO] Benchmarking: GaussianCopula
2026-08-05 20:06:06,383 [INFO]   Seed 42 ...
2026-08-05 20:06:06,392 [INFO]     Fitting GaussianCopula on 8000 rows...
2026-08-05 20:06:06,443 [INFO] Detected metadata:
2026-08-05 20:06:06,443 [INFO] {
    "METADATA_SPEC_VERSION": "SINGLE_TABLE_V1",
    "columns": {
        "transaction_date": {
            "datetime_format": "%Y-%m-%d",
            "sdtype": "datetime"
        },
        "transaction_time": {
            "sdtype": "categorical"
        },
        "transaction_type": {
            "sdtype": "categorical"
        },
        "amount_npr": {
            "sdtype": "numerical"
        },
        "exchange_rate": {
            "sdtype": "numerical"
        },
        "channel": {
            "sdtype": "categorical

PerformanceAlert: Using the CTGANSynthesizer on this data is not recommended. To model this data, CTGAN will generate a large number of columns.

Original Column Name      Est # of Columns (CTGAN)
transaction_date          11
transaction_time          7530
transaction_type          4
amount_npr                11
exchange_rate             11
channel                   4
sender_country            18
receiver_country          18
is_cross_border           2
remittance_corridor       48
merchant_category         1
device_type               4
ip_country                18
ip_is_vpn                 1
is_fraud                  2
fraud_type                4
aml_risk_indicator        2
hour_of_day               11
day_of_week               7
is_weekend                2
month                     2
above_1M_NPR              2
above_10M_NPR             2
velocity_sum_10tx         11
tx_count_10               11
tx_count_30               11
currency_mismatch         2
sender_country_risk       11
rece

2026-08-05 20:15:09,759 [INFO] Guidance: There are no missing values in column transaction_date. Extra column not created.
2026-08-05 20:15:09,839 [INFO] Guidance: There are no missing values in column amount_npr. Extra column not created.
2026-08-05 20:15:10,216 [INFO] Guidance: There are no missing values in column exchange_rate. Extra column not created.
2026-08-05 20:15:10,292 [INFO] Guidance: There are no missing values in column hour_of_day. Extra column not created.
2026-08-05 20:15:10,686 [INFO] Guidance: There are no missing values in column velocity_sum_10tx. Extra column not created.
2026-08-05 20:15:11,070 [INFO] Guidance: There are no missing values in column tx_count_10. Extra column not created.
2026-08-05 20:15:11,424 [INFO] Guidance: There are no missing values in column tx_count_30. Extra column not created.
2026-08-05 20:15:11,794 [INFO] Guidance: There are no missing values in column sender_country_risk. Extra column not created.
2026-08-05 20:15:11,814 [INFO] Guida

PerformanceAlert: Using the CTGANSynthesizer on this data is not recommended. To model this data, CTGAN will generate a large number of columns.

Original Column Name      Est # of Columns (CTGAN)
transaction_date          11
transaction_time          7545
transaction_type          4
amount_npr                11
exchange_rate             11
channel                   4
sender_country            18
receiver_country          18
is_cross_border           2
remittance_corridor       46
merchant_category         1
device_type               4
ip_country                18
ip_is_vpn                 1
is_fraud                  2
fraud_type                4
aml_risk_indicator        2
hour_of_day               11
day_of_week               7
is_weekend                2
month                     2
above_1M_NPR              2
above_10M_NPR             2
velocity_sum_10tx         11
tx_count_10               11
tx_count_30               11
currency_mismatch         2
sender_country_risk       11
rece

2026-08-05 20:21:00,845 [INFO] Guidance: There are no missing values in column exchange_rate. Extra column not created.
2026-08-05 20:21:01,024 [INFO] Guidance: There are no missing values in column hour_of_day. Extra column not created.
2026-08-05 20:21:01,404 [INFO] Guidance: There are no missing values in column velocity_sum_10tx. Extra column not created.
2026-08-05 20:21:01,806 [INFO] Guidance: There are no missing values in column tx_count_10. Extra column not created.
2026-08-05 20:21:02,176 [INFO] Guidance: There are no missing values in column tx_count_30. Extra column not created.
2026-08-05 20:21:02,554 [INFO] Guidance: There are no missing values in column sender_country_risk. Extra column not created.
2026-08-05 20:21:02,585 [INFO] Guidance: There are no missing values in column receiver_country_risk. Extra column not created.
2026-08-05 20:21:02,629 [INFO] Guidance: There are no missing values in column sender_account_age_days. Extra column not created.
2026-08-05 20:21:0

...

PerformanceAlert: Using the CTGANSynthesizer on this data is not recommended. To model this data, CTGAN will generate a large number of columns.

Original Column Name      Est # of Columns (CTGAN)
transaction_date          11
transaction_time          7545
transaction_type          4
amount_npr                11
exchange_rate             11
channel                   4
sender_country            18
receiver_country          18
is_cross_border           2
remittance_corridor       46
merchant_category         1
device_type               4
ip_country                18
ip_is_vpn                 1
is_fraud                  2
fraud_type                4
aml_risk_indicator        2
hour_of_day               11
day_of_week               7
is_weekend                2
month                     2
above_1M_NPR              2
above_10M_NPR             2
velocity_sum_10tx         11
tx_count_10               11
tx_count_30               11
currency_mismatch         2
sender_country_risk       11
rece

2026-08-05 20:46:10,756 [INFO] Guidance: There are no missing values in column exchange_rate. Extra column not created.
2026-08-05 20:46:10,847 [INFO] Guidance: There are no missing values in column hour_of_day. Extra column not created.
2026-08-05 20:46:10,936 [INFO] Guidance: There are no missing values in column velocity_sum_10tx. Extra column not created.
2026-08-05 20:46:11,037 [INFO] Guidance: There are no missing values in column tx_count_10. Extra column not created.
2026-08-05 20:46:11,080 [INFO] Guidance: There are no missing values in column tx_count_30. Extra column not created.
2026-08-05 20:46:11,170 [INFO] Guidance: There are no missing values in column sender_country_risk. Extra column not created.
2026-08-05 20:46:11,269 [INFO] Guidance: There are no missing values in column receiver_country_risk. Extra column not created.
2026-08-05 20:46:11,364 [INFO] Guidance: There are no missing values in column sender_account_age_days. Extra column not created.
2026-08-05 20:46:1

PerformanceAlert: Using the CTGANSynthesizer on this data is not recommended. To model this data, CTGAN will generate a large number of columns.

Original Column Name      Est # of Columns (CTGAN)
transaction_date          11
transaction_time          7544
transaction_type          4
amount_npr                11
exchange_rate             11
channel                   4
sender_country            18
receiver_country          18
is_cross_border           2
remittance_corridor       51
merchant_category         1
device_type               4
ip_country                18
ip_is_vpn                 1
is_fraud                  2
fraud_type                4
aml_risk_indicator        2
hour_of_day               11
day_of_week               7
is_weekend                2
month                     2
above_1M_NPR              2
above_10M_NPR             2
velocity_sum_10tx         11
tx_count_10               11
tx_count_30               11
currency_mismatch         2
sender_country_risk       11
rece

2026-08-05 20:51:56,905 [INFO] Guidance: There are no missing values in column exchange_rate. Extra column not created.
2026-08-05 20:51:56,983 [INFO] Guidance: There are no missing values in column hour_of_day. Extra column not created.
2026-08-05 20:51:57,074 [INFO] Guidance: There are no missing values in column velocity_sum_10tx. Extra column not created.
2026-08-05 20:51:57,173 [INFO] Guidance: There are no missing values in column tx_count_10. Extra column not created.
2026-08-05 20:51:57,216 [INFO] Guidance: There are no missing values in column tx_count_30. Extra column not created.
2026-08-05 20:51:57,285 [INFO] Guidance: There are no missing values in column sender_country_risk. Extra column not created.
2026-08-05 20:51:57,383 [INFO] Guidance: There are no missing values in column receiver_country_risk. Extra column not created.
2026-08-05 20:51:57,433 [INFO] Guidance: There are no missing values in column sender_account_age_days. Extra column not created.
2026-08-05 20:51:5


✓ Benchmark complete


In [ ]:
from src.evaluation.benchmark_runner import BenchmarkRunner

runner = BenchmarkRunner(
    real_data=ml_data,
    target_col=TARGET_COL,
    seeds=[42,4,6],
    test_size=0.2,
    output_dir=OUTPUT_DIR
)

my_synthesizer={"TabSyn":TabSynGenerator(config={"epoch":100}),
                    "COPULA"
            }
results = runner.run(TabSynGenerator(config={"epoch":100}))


2026-08-05 21:33:32,812 [INFO] BenchmarkRunner: 10000 rows, target=is_suspicious_tx, 9 continuous, 48 categorical


AttributeError: 'TabSynGenerator' object has no attribute 'items'

## 4. Final Comparison Report

In [6]:
from src.evaluation.report import build_final_table

# Build and save the final table (CSV + Markdown)
final_df = build_final_table(results, output_dir=OUTPUT_DIR)

# Display nicely in the notebook
final_df

2026-08-05 21:12:37,341 [INFO] 
| Method | Accuracy (%) | F1-score | AUC | Avg JSD | Avg WD | Diff. Corr. |
| --- | --- | --- | --- | --- | --- | --- |
| GaussianCopula | 0.4125 ± 0.2034 | 0.0041 ± 0.0011 | 0.2738 ± 0.0093 | 0.1087 ± 0.0043 | 0.0206 ± 0.0044 | 11.3812 ± 0.3552 |
| CTGAN | 2.0042 ± 1.4203 | 0.0124 ± 0.0078 | 0.1543 ± 0.0770 | 0.1079 ± 0.0077 | 0.0486 ± 0.0114 | 12.3747 ± 0.1319 |
| TVAE | 0.0000 ± 0.0000 | 0.0000 ± 0.0000 | 0.0877 ± 0.0889 | 0.1392 ± 0.0004 | 0.0797 ± 0.0151 | 9.4143 ± 0.0410 |
| CopulaGAN | 1.3083 ± 0.5613 | 0.0086 ± 0.0029 | 0.1710 ± 0.0274 | 0.1135 ± 0.0029 | 0.0520 ± 0.0128 | 12.4670 ± 0.0426 |
| TabDDPM | 0.4542 ± 0.2974 | 0.0041 ± 0.0017 | 0.2210 ± 0.0740 | 0.0847 ± 0.0085 | 0.4026 ± 0.0006 | 63713402.8137 ± 90104341.2039 |

2026-08-05 21:12:37,369 [INFO] Saved final_table.csv → /home/red/Desktop/neural-sentinel/docs/generator_benchmark/final_table.csv
2026-08-05 21:12:37,384 [INFO] Saved final_table.md → /home/red/Desktop/neural-sentinel/docs/gen

,Accuracy (%),F1-score,AUC,Avg JSD,Avg WD,Diff. Corr.
Method,,,,,,
GaussianCopula,0.4125 ± 0.2034,0.0041 ± 0.0011,0.2738 ± 0.0093,0.1087 ± 0.0043,0.0206 ± 0.0044,11.3812 ± 0.3552
CTGAN,2.0042 ± 1.4203,0.0124 ± 0.0078,0.1543 ± 0.0770,0.1079 ± 0.0077,0.0486 ± 0.0114,12.3747 ± 0.1319
TVAE,0.0000 ± 0.0000,0.0000 ± 0.0000,0.0877 ± 0.0889,0.1392 ± 0.0004,0.0797 ± 0.0151,9.4143 ± 0.0410
CopulaGAN,1.3083 ± 0.5613,0.0086 ± 0.0029,0.1710 ± 0.0274,0.1135 ± 0.0029,0.0520 ± 0.0128,12.4670 ± 0.0426
TabDDPM,0.4542 ± 0.2974,0.0041 ± 0.0017,0.2210 ± 0.0740,0.0847 ± 0.0085,0.4026 ± 0.0006,63713402.8137 ± 90104341.2039
